In [1]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional, Dict, Any, List

In [ ]:
@dataclass
class PetProfile:
    pet_id: str
    pet_name: str
    breed: str                      # for searching any special attribute of the breed
    birth_date: datetime
    is_high_energy: bool            # e.g., Vizsla 
    postcode: str                   # For seasonal/geographic tick risk 
    streak: int                     # auto calculate number of checkin in (consecutive), reset if hasn't checkin more than 1 week (7 days)
    consistency: float              # Automatic check if has owner has checkin regularly
    worming_compliant: bool         # Non-compliance moves score to Action Needed 
    vaccination_current: bool       # Treatment currency feeds score 
    days_since_last_vet_visit: int   # Auto-calculated metric 
    
    pet_name: Optional[str] = None
    owner_name: Optional[str] = None


In [6]:
@dataclass
class VetInteractionLog:
    """Vet log after checkin"""
    visited_vet: bool
    reason_for_visit: Optional[str] = None # Categorized reason informs score 
    log: Optional[str] = None




In [7]:
@dataclass
class DentalHealthLog:
    """Periodic check-in (Puppy teething, 6-month baseline, and 6-monthly ongoing)"""
    breath_odour: str             # e.g., "Noticeably worse", "Normal" 
    chewing_changes: str          # e.g., "Less interested", "Favouring one side" 
    gum_redness_bleeding: str     # e.g., "Noticed buildup/redness", "Normal" 
    professional_clean_due: bool  # True/False

In [8]:
@dataclass
class ToiletingLog:
    """Tracks bowel habits, frequency, and acute overrides"""
    frequency_changes: str    # e.g., "Normal", "More frequent", "Less frequent"
    consistency_changes: str  # e.g., "Normal", "Runny", "Hard" 
    indoor_accidents: bool    # True/False 
    blood_present: bool

In [ ]:
@dataclass
class CheckInData:
    """The Weekly Check-in Data payload"""
    pet_id: str
    timestamp: datetime
    
    # Pre-Moment 1 Context
    vet_interaction: VetInteractionLog 
    
    # Core Weekly Vitals (Moment 1 & 2)
    energy: int                  # 1-5 scale (comparative to baseline) 
    appetite: int                # 1-5 scale 
    sleep_quality: int           # 1-5 scale
    activity_score: int         # 1-5 scale
    water_intake: str            # "Normal", "Increased", "Decreased" (Increased = High Risk) 
    
    
    # Behaviour & Changes
    diet_changed: bool           # Elevated if coincides with health signal 
    social_behaviour: str        # e.g., "Normal", "Withdrawn", "Clingy", "Aggressive" 
    
    # Physical & Specific Logs
    physical_health: Dict[str, int] # e.g., {"eyes": 10, "ears": 8, "coat": 9} 
    toileting_status: ToiletingLog         # Modular sub-class 
    
    # Parasite & Pain Checks (Moment 1 Additions)
    blood_in_stool: bool
    vomitting: bool
    diarrhea: bool
    breathing_difficulty: bool
    collapsed: bool
    seizure: bool
    suspected_toxin_ingestion: bool
    urinate_difficulty: bool
    severe_bleeding: bool
    tick_found: bool
    pain_signs: bool

    # Periodic & Monthly Inputs (Can be None/Null on standard weeks)
    weight: Optional[float] = None              # First check-in of each month 
    dental_log: Optional[DentalHealthLog] = None # Periodic (weeks 12-24, or 6-monthly)
    
    def blood_in_stool(self) -> bool:
        """Auto flag if blood is present in stool, overrides owner input for accuracy"""
        self.blood_in_stool = ToiletingLog.blood_present
        return self.blood_in_stool
    
    

In [11]:
@dataclass
class RedflagData:
    """Red-flag conditions that trigger immediate alerts"""
    blood_in_stool: bool
    vomitting: bool
    diarrhea: bool
    breathing_difficulty: bool
    collapsed: bool
    seizure: bool
    suspected_toxin_ingestion: bool
    urinate_difficulty: bool
    severe_bleeding: bool
    tick_found: bool
    pain_signs: bool

In [12]:
@dataclass
class VitalityScoreOutput:
    """Output"""
    vitality_score: Optional[int]
    zone: str
    trend: str
    confidence: str
    top_drivers: List[str]
    red_flag: bool
    recommended_next_step: str
    owner_summary: str


In [ ]:
class VitalityScoreEngine:
    """
    Layer 1: Calculates the 0-100 wellness score based on comparative baseline logic and dynamic breed weighting.
    """
    def __init__(self, pet: PetProfile, history: list[CheckInData]):
        self.pet = pet
        self.history = history # Previous 4 weeks for baseline (should use only 4 previous week)

    def calculate_score(self, current: CheckInData) -> VitalityScoreOutput:
        
        # Acute Overrides
        is_sudden_collapse = (current.energy <= 2 and current.appetite <= 2)

        if current.toileting.blood_present or current.water_intake == "Increased" or is_sudden_collapse:
            return self._generate_action_needed_override(current)
        
        # Checking if meet the requirement length of the length is met
        if len(self.history) < 4:
            return VitalityScoreOutput(
                vitality_score=None,
                zone="Building Baseline",
                trend="N/A",
                confidence="Low",
                top_drivers=["Establishing normal patterns"],
                red_flag=False,
                recommended_next_step="Continue completing weekly check-ins.",
                owner_summary="Evie is still learning what is normal for your pet during these first four weeks."
            )
        
        # Average history baseline
        avg_energy = sum(h.energy for h in self.history) / 4.0
        avg_appetite = sum(h.appetite for h in self.history) / 4.0
        avg_sleep = sum(h.sleep_quality for h in self.history) / 4.0
        avg_act = max(1, sum(h.activity_minutes for h in self.history) / 4.0)

        # Calculate Negative Deviations
        drop_energy = max(0, avg_energy - current.energy)                 # Energy
        drop_appetite = max(0, avg_appetite - current.appetite)           # Appetite
        drop_sleep = max(0, avg_sleep - current.sleep_quality)            # Sleep
        drop_act = max(0, (avg_act - current.activity_minutes) / avg_act) # Activity minute

        # Weighting (Normal breed v.s High Energy breed)
        w_act, w_sleep, w_energy, w_appetite = 0.25, 0.25, 0.25, 0.25
        if self.pet.is_high_energy:
            w_act = 0.40
            w_sleep = 0.10

        raw_score = 100.0
        drivers = []

        # Sighal v.s Noise
        if drop_act >= 1.0 and (drop_energy >= 1.0 or drop_sleep >= 1.0):
            raw_score -= 25.0
            drivers.append("Combined drop in appetite and energy/sleep")
        else:
            raw_score -= (drop_appetite * 10 * w_appetite)
            raw_score -= (drop_energy * 10 * w_energy)
            raw_score -= (drop_sleep * 10 * w_sleep)
            if drop_act  >= 1: drivers.append("Slightly lower appetite")
            if drop_energy >= 1: drivers.append("Slightly lower energy")

        # Activity Penalty
        if drop_act > 0.15:
            raw_score -= (drop_act * 30 * w_act)
            drivers.append("Lower activity than usual")

        # Medium weight vitals, platform data
        if current.social_behaviour in ["Withdrawn", "Aggressive", "Clingy"]:
            raw_score -= 10.0
            drivers.append(f"Social behaviour shift ({current.social_behaviour})")
            
        if not self.pet.worming_compliant:
            raw_score -= 10.0
            drivers.append("Overdue for worming treatment")

        final_score = max(0, min(100, int(raw_score)))
        zone_name = self._get_zone(final_score)
        
        if not drivers:
            drivers = ["Vitals are stable and consistent"]

        return VitalityScoreOutput(
            vitality_score=final_score,
            zone=zone_name,
            trend="Stable" if final_score >= 85 else "Slightly down", 
            confidence=self.pet.consistency,
            top_drivers=drivers,
            red_flag=False,
            recommended_next_step=self._get_next_step(zone_name),
            owner_summary=self._generate_summary(zone_name, drivers)
        )
    
    def _get_zone(self, score: int) -> str:
        if score >= 85: return "Bright Green"
        if score >= 70: return "Medium Green"
        if score >= 50: return "Watch Zone"
        return "Action Needed"

    def _get_next_step(self, zone: str) -> str:
        if zone in ["Bright Green", "Medium Green"]: return "Keep up the great routine."
        if zone == "Watch Zone": return "Monitor closely; consider reviewing our vet directory."
        return "Contact your veterinarian for advice." 

    def _generate_summary(self, zone: str, drivers: List[str]) -> str:
        if zone == "Bright Green":
            return "Your pet is doing wonderfully and tracking perfectly with their normal baseline."
        return f"Your pet seems a little below their usual pattern this week, mainly due to {drivers[0].lower()}."
